In [1]:
!pip install -q kaggle

In [ ]:
import os
import json

os.makedirs("/root/.kaggle", exist_ok=True)

kaggle_credentials = {
   "username":"",
"key":""
}

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)

os.chmod("/root/.kaggle/kaggle.json", 600)

In [3]:
!mkdir -p /content/data

!kaggle datasets download -d jangedoo/utkface-new\
    -p /content/data \
    --unzip

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
100% 331M/331M [00:04<00:00, 75.4MB/s] 



In [4]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [5]:
folder_path = '/content/data/UTKFace'

In [6]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [7]:
len(age)

23708

In [8]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [9]:
df.shape

(23708, 3)

In [10]:
df.head()

,age,gender,img
0,26,1,26_1_3_20170104232510106.jpg.chip.jpg
1,30,1,30_1_0_20170109012829305.jpg.chip.jpg
2,28,1,28_1_1_20170113012524543.jpg.chip.jpg
3,76,0,76_0_2_20170112224304323.jpg.chip.jpg
4,34,1,34_1_0_20170116160755190.jpg.chip.jpg


In [11]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [12]:
train_df.shape

(20000, 3)

In [13]:
test_df.shape

(3708, 3)

In [14]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [15]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [16]:
def generator_wrapper(generator):
    while True:
        x, y = next(generator)
        yield x, (y[0], y[1])

In [17]:
train_data = generator_wrapper(train_generator)
test_data = generator_wrapper(test_generator)

In [18]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [19]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [20]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [21]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [22]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [23]:
history = model.fit(
    train_data,
    steps_per_epoch=len(train_generator),
    epochs=10,
    validation_data=test_data,
    validation_steps=len(test_generator)
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 249s 376ms/step - age_loss: 15.4314 - age_mae: 15.4314 - gender_accuracy: 0.5163 - gender_loss: 0.8387 - loss: 98.4632 - val_age_loss: 14.7612 - val_age_mae: 14.7600 - val_gender_accuracy: 0.5181 - val_gender_loss: 0.6926 - val_loss: 83.3228
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 229s 367ms/step - age_loss: 15.0352 - age_mae: 15.0352 - gender_accuracy: 0.5229 - gender_loss: 0.6926 - loss: 83.6005 - val_age_loss: 14.9773 - val_age_mae: 14.9773 - val_gender_accuracy: 0.5178 - val_gender_loss: 0.6925 - val_loss: 83.5382
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 236s 379ms/step - age_loss: 14.9310 - age_mae: 14.9310 - gender_accuracy: 0.5235 - gender_loss: 0.6927 - loss: 83.5065 - val_age_loss: 14.4660 - val_age_mae: 14.4660 - val_gender_accuracy: 0.5167 - val_gender_loss: 0.6926 - val_loss: 83.0377
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 233s 372ms/step - age_loss: 14.8202 - age_mae: 14.8202 - gender_accuracy: 0.5239 - gender_loss: 0.6954 - loss: 83.